# Revision: Reading Messy JSON

This notebook covers working with real, nested Twitter-data-shaped JSON (the format produced by `twarc`, the tool used earlier in the course to collect Twitter data), plus the conceptual "why" questions several of you raised in the Week 3 discussion (why merge is required, why pseudonymize, how to make open-ended decisions, how to judge sentiment disagreements) and a couple of practical aspect.

## Setup

In [3]:
import pandas as pd
import re
import json

# How is JSON different from a DataFrame?

Before opening the real file, let's build up the idea with a tiny made-up example.

**JSON** is a way of writing data as plain *text*, so it can be sent from one computer program to another (for example, from Twitter's servers to your laptop). Since you are familiar with Python dictionary before, JSON will look familiar: it uses the same `{key: value}` style, just written out as one long piece of text instead of being a "live" Python object in memory yet.

Here's a tiny JSON example, written as a Python string so you can see what the raw text looks like:


In [18]:
json_text = '{"id": "123", "text": "Hello world!", "public_metrics": {"like_count": 5, "retweet_count": 1}}'
print(json_text)
print(type(json_text))

{"id": "123", "text": "Hello world!", "public_metrics": {"like_count": 5, "retweet_count": 1}}
<class 'str'>


Right now, `json_text` is just one long string. Python doesn't yet know it *means* a dictionary with an `id`, a `text`, and so on — you can't write `json_text['text']`, because to Python it's still just text, like a sentence in an email.

The `json` module's `.loads()` function ("load string") is what turns that text into an actual, usable Python dictionary:


In [5]:
tweet_dict = json.loads(json_text)
print(tweet_dict)
print(type(tweet_dict))
tweet_dict['text']

{'id': '123', 'text': 'Hello world!', 'public_metrics': {'like_count': 5, 'retweet_count': 1}}
<class 'dict'>


'Hello world!'

Notice that `public_metrics` is a dictionary sitting *inside* the outer dictionary. This is the key difference to keep in mind for the rest of this notebook:

* A **DataFrame** is a flat table: every row has the same columns, and each cell normally holds one simple value (a number, a bit of text, a date).
* **JSON** doesn't have to be flat: one field can hold a whole list, or another dictionary, or a list of dictionaries.

In [6]:
jsonl_text = '{"id": "1", "text": "First tweet"}\n{"id": "2", "text": "Second tweet"}'
print(jsonl_text)

{"id": "1", "text": "First tweet"}
{"id": "2", "text": "Second tweet"}


Notice there's no `,` joining the two objects and no `[ ]` wrapping them the way a Python list would have.

In [7]:
lines = jsonl_text.splitlines()          # split the text into a list of lines
parsed = [json.loads(line) for line in lines]   # turn each line into its own dictionary
parsed

[{'id': '1', 'text': 'First tweet'}, {'id': '2', 'text': 'Second tweet'}]

That loop is the manual, do-it-yourself version of what `pd.read_json(..., lines=True)` does for you automatically: it reads the file one line at a time, and parses each line as its own separate JSON object. Now that you've seen *why* that's necessary, the next question — and the error it deliberately walks you into — will make a lot more sense.


# Part 1 — Reading nested JSON

## Question 1 — Load the file 

Try loading `synthetic_tweets_raw.jsonl` with `pd.read_json('da3_data/synthetic_tweets_raw.jsonl')`. Read the error message. Then fix it by adding `lines=True`.

**Why this matters:**  A `.jsonl` file is not one big JSON document — it's many separate JSON objects, one per line (that's what the 'L' means). `pd.read_json()` assumes a single document by default, so it parses the first line successfully, then finds more data after it and gives up. `lines=True` tells it to parse the file line-by-line instead. Once you know this, the error stops being scary and becomes a one-argument fix.

**Hint:**
```python
df_jsonl = pd.read_json('da3_data/synthetic_tweets_raw.jsonl')  # error
df_jsonl = pd.read_json('da3_data/synthetic_tweets_raw.jsonl', lines=True)  # fixed
```

## Question 2 — Inspect the raw (still nested) structure

Look at `df_jsonl.columns`, `df_jsonl.dtypes`, and then `df_jsonl['data'].iloc[0]` — the first row's `data` value. What Python type is it? What's inside it?

**Why this matters:** This is the step people skip, and it's the one that actually builds understanding. `df_jsonl` loaded successfully, but it is **not** a tidy tweet-per-row table yet — each row is one API 'page,' and the actual tweets are buried inside a Python list (inside the `data` column) of dictionaries. Nothing in `df_jsonl` is directly usable for analysis (you can't `.groupby()` a column full of lists). Seeing this with your own eyes — instead of just running someone else's flattening code — is what makes the *next* question make sense instead of feeling like magic.

**Hint:**
```python
df_jsonl.dtypes
type(df_jsonl['data'].iloc[0])
df_jsonl['data'].iloc[0][0]  # the first tweet dict on the first page
```